In [1]:
# !pip install langchain_huggingface
# !pip install sentence-transformers

In [ ]:
from __future__ import annotations

from typing import List, Dict, Any, Optional
import os
import asyncio

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from src.faiss_storage import FaissStorage


EMB_MODEL_NAME = "BAAI/bge-small-en-v1.5"
EMB_DIM = 384
CHUNK_SIZE=512
CHUNK_OVERLAP=200

embed_model = HuggingFaceEmbeddings(model_name=EMB_MODEL_NAME)
splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)


# load the pdf file from the path and chunk it
def load_and_chunk_pdf(pdf_path: str) -> List[Document]:
    loader = PyPDFLoader(pdf_path)
    text = loader.load()
    text = [d.page_content for d in text if getattr(d, "page_content", None)]

    chunks = []
    for t in text:
        chunks.extend(splitter.split_text(t))
    
    return chunks



def embed_texts(texts: list[str]) -> list[list[float]]:
    response = embed_model.embed_documents(texts)
    return response

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
pdf_path = r"C:\\Users\\yashd\\OneDrive\\Desktop\\updated cvs\\UK resumes\\yash_jobster.pdf"

pages = load_and_chunk_pdf(pdf_path)
text_embs = embed_texts(pages)

store = FaissStorage().upsert(ids=[i for i in range(len(pages))], vectors=text_embs, payloads=[{"text": pages[i], "source": pdf_path} for i in range(len(pages))])  

In [8]:
from src.faiss_storage import FaissStorage
from src.ingest_pdf import embed_texts
store = FaissStorage()

query = "who is yash"
text_emb = embed_texts(query)[0]

result = store.search(
    query_vector=text_emb,
    top_k=5
)

result

{'contexts': ['Yash Dhakade \nGlasgow, UK \n+44 7823703834, \nyashdhakade2021@gmail.com \nwww.linkedin.com/in/yashdhakade \nSUMMARY \nApplied AI Engineer with hands-on experience building and deploying Python-based ML and LLM systems used in \nreal-world workflows. Strong background across the full AI lifecycle, including data pipeline design, model training and \nevaluation, API-backed services, and deployment-ready ML systems. Experienced with Retrieval-Augmented',
  'analysis, feature engineering \nGenerative AI & LLMs: Retrieval-Augmented Generation (RAG), LangChain, embeddings, prompt pipelines, \nbias evaluation \nMLOps & Deployment: Docker, AWS (EC2, S3), model evaluation, experiment tracking, reproducible \npipelines \nData & Search Systems: FAISS, vector databases, large-scale document retrieval',
  'Quantitative Methods for Artificial Intelligence. \nDissertation: Experimental evaluation of NLP and Large Language Model systems, with emphasis on \nbias detection, robustness an

## load jsonl file

In [51]:
from __future__ import annotations

from typing import List, Dict, Any, Optional
import os
import asyncio
import orjson
from tqdm import tqdm

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from src.faiss_storage import FaissStorage


EMB_MODEL_NAME = "BAAI/bge-small-en-v1.5"
EMB_DIM = 384
CHUNK_SIZE=512
CHUNK_OVERLAP=200

embed_model = HuggingFaceEmbeddings(model_name=EMB_MODEL_NAME)
splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)


# load the pdf file from the path and chunk it
def stream_jsonl(path: str) -> Iterator[Dict[str, Any]]:
    with open(path, "rb") as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                yield orjson.loads(line)
            except orjson.JSONDecodeError as e:
                raise ValueError(f"Bad JSON on line {line_no}: {e}") from e


# chunk the text 
# def chunk_text(text: str, chunk_size: int = 900, overlap: int = 150, min_chars: int = 200) -> List[str]:
#     text = text.strip()
#     if not text:
#         return []
#     step = max(1, chunk_size - overlap)
#     chunks = []
#     for start in range(0, len(text), step):
#         end = min(len(text), start + chunk_size)
#         ch = text[start:end].strip()
#         if len(ch) >= min_chars:
#             chunks.append(ch)
#         if end == len(text):
#             break
#     return chunks

def chunk_text(
    text: str,
    chunk_size: int = 900,
    overlap: int = 150,
    min_chars: int = 200,
):
    text = text.strip()
    if not text:
        return []

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=overlap,
        separators=["\n\n", "\n", " ", ""],  # paragraph → line → word → char
        length_function=len,
    )

    chunks = splitter.split_text(text)

    # Optional: enforce minimum length (like your current logic)
    chunks = [c.strip() for c in chunks if len(c.strip()) >= min_chars]

    return chunks

def build_payloads(rec: Dict[str, Any], chunks: List[str]) -> List[Dict[str, Any]]:
    doc_id = str(rec.get("doc_id", ""))
    title = str(rec.get("title", ""))
    collection = str(rec.get("collection", ""))
    source_dir = str(rec.get("source_dir", ""))
    document_type = str(rec.get("document_type", ""))

    payloads = []
    for i, ch in enumerate(chunks):
        payloads.append({
            "text": ch,
            "source": source_dir or doc_id,     # what you want to show as citation/source
            "doc_id": doc_id,
            "chunk_id": i,
            "title": title,
            "document_type": document_type,
            "collection": collection,
            "date": rec.get("date", ""),
            "date_numeric": rec.get("date_numeric", None),
            "ocr_quality": rec.get("ocr_quality", None),
            "ocr_quality_tier": rec.get("ocr_quality_tier", None),
        })
    return payloads



# def embed_texts(texts: list[str]) -> list[list[float]]:
#     response = embed_model.embed_documents(texts)
#     return response

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
def load_and_chunk_jsonl(jsonl_path: str):

    next_id = 0
    min_ocr_quality: float = 0.0

    ids_buf: List[int] = []
    texts_buf: List[str] = []
    payloads_buf: List[Dict[str, Any]] = []

    for line in tqdm(stream_jsonl(jsonl_path), desc=f"Ingest Jsonl File: {jsonl_path}"):
        # Optional OCR quality filter
        oq = line.get("ocr_quality", None)
        if oq is not None:
            try:
                if float(oq) < float(min_ocr_quality):
                    continue
            except Exception:
                pass

        chunks = chunk_text(line["text"])
        payloads = build_payloads(line, chunks=chunks)
        # print(payloads)

        for ch, payload in zip(chunks, payloads):
            ids_buf.append(next_id)
            texts_buf.append(ch)
            payloads_buf.append(payload)
            next_id += 1

In [ ]:
path_to_corpus = "data/corpus2.jsonl"
next_id = 0
ids_buf: List[int] = []
texts_buf: List[str] = []
payloads_buf: List[Dict[str, Any]] = []

min_ocr_quality: float = 0.0
for line in tqdm(stream_jsonl(path_to_corpus), desc="Ingest corpus.jsonl"):
        # Optional OCR quality filter
        oq = line.get("ocr_quality", None)
        if oq is not None:
            try:
                if float(oq) < float(min_ocr_quality):
                    continue
            except Exception:
                pass

        chunks = chunk_text(line["text"])
        payloads = build_payloads(line, chunks=chunks)
        # print(payloads)

        text = rec.get("text", "")
        if not text:
            continue

        print(text)

Ingest corpus.jsonl: 0it [00:00, ?it/s]

Cauint 387723 (18
Battles a horstan

D. 675/23
Battles a Kordofan
Beiter Havans Balid at Shatzen und fligh
From Repal's
Ovarians
Junal at Pass. Oriens par Kordofan: 1590.
Battle of the Haraz: 1 Su 1299
While Bimeb . NAZIM Eff. was collesting the govt . Lanes al
Tamra
UMM SAAR0 g the Grabo, we him ABorecast W. Ar
NVA same with a few men from MUMD. Arts and began to
icile We Arabs to disobedience . The grabs rese against
Naguis Eff., are up the registers and stole the money . Nagin
Eff. was compelled to retime to the Haraf with the Sheilkes
We Hamas , IBM Bey An-Minist and the An-RAsim An
DARAL , as an escort ' since he had all one company a licops
with hin.
When Naguis Eff. receded the Manag , he found the
BIOTRIYAH grabs in revolt against the Nazir , Mur3 . Aglea
RAAMAN . So he wrote at one is 21 Obeid for reinforcements
Siren. Mur3 . Aglea Ar0 BALLA came with a forreg 150
inegulars and s0 boxes of ammunition . Meanwhile the Hamas
had journed the Bidiminals, we find being to be marks w

In [1]:
from src.ingest_corpus_jsonl import load_and_chunk_jsonl, embed_texts
from src.faiss_storage import FaissStorage


path_to_corpus = "data/corpus2.jsonl"
chunks_with_payloads = load_and_chunk_jsonl(path_to_corpus)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
from src.custom_types import RAGChunkAndSrc, RAGUpsertResult, RAGSearchResult, RAQQueryResult
import uuid
import os

def _load(jsonl_path) -> RAGChunkAndSrc:
    # jsonl_path = ctx.event.data["jsonl_path"]
    # source_id = ctx.event.data.get("source_id", jsonl_path)

    # Stream records, clean, chunk per record (keep metadata!)
    chunks = load_and_chunk_jsonl(jsonl_path)
    return RAGChunkAndSrc(chunks=chunks, source_id="source_id")


def _upsert(chunk_and_src: RAGChunkAndSrc) -> RAGUpsertResult:
    chunks = chunk_and_src.chunks
    source_id = chunk_and_src.source_id
    
    texts = [c["text"] for c in chunks]
    vecs = embed_texts(texts)
    ids = [str(uuid.uuid5(uuid.NAMESPACE_URL, f"{source_id}:{i}")) for i in range(len(chunks))]
    # payloads = [{"source": source_id, "text": c["text"]} for c in chunks]
    payloads = []
    for i, c in enumerate(chunks):
        payload = {"source": source_id, "text": c["text"]}
        payload.update(c.get("metadata", {}))
        payloads.append(payload)
    FaissStorage().upsert(ids, vecs, payloads)
    return RAGUpsertResult(ingested=len(chunks))

In [3]:
chunks = _load(path_to_corpus)
upsert_result = _upsert(chunks)

NameError: name '_load' is not defined

In [2]:
from src.custom_types import RAGChunkAndSrc, RAGUpsertResult, RAGSearchResult, RAQQueryResult
import uuid
import os
from src.ingest_corpus_jsonl import load_and_chunk_jsonl, embed_texts
from src.faiss_storage import FaissStorage


path_to_corpus = "data/corpus2.jsonl"

def _process_corpus(jsonl_path) -> RAGUpsertResult:
    # jsonl_path = ctx.event.data["jsonl_path"]
    # source_id = ctx.event.data.get("source_id", jsonl_path)

    source_id = jsonl_path

    # Stream records, clean, chunk per record (keep metadata!)
    chunks = load_and_chunk_jsonl(jsonl_path)
    
    texts = [c["text"] for c in chunks]
    vecs = embed_texts(texts)
    ids = [str(uuid.uuid5(uuid.NAMESPACE_URL, f"{source_id}:{i}")) for i in range(len(chunks))]
    
    payloads = []
    for i, c in enumerate(chunks):
        payload = {"source": source_id, "text": c["text"]}
        payload.update(c.get("metadata", {}))
        payloads.append(payload)
        
    FaissStorage().upsert(ids, vecs, payloads)
    return RAGUpsertResult(ingested=len(chunks))

In [3]:
results = _process_corpus(path_to_corpus)


In [ ]:
results.searc